In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

In [3]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
import json
from evaluation_utils import llm_structured

for doc in documents[:3]:
    user_prompt = json.dumps(doc)
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    print(result.questions)
    print(usage.input_tokens)

['What is a Retrieval-Augmented Generation system, and how does it help when an LLM may not know the answer on its own?', 'Why does this course treat LLMs like black boxes instead of explaining how they work internally?', 'What are the main weaknesses of LLMs that RAG is meant to fix?', 'What is the FAQ agent in this module supposed to do, and what kind of question should it answer?', 'What will the first part of the module build, and how is the second part different?']
1020
['What do I need installed before starting this module besides Python, and which Python version should I have?', 'How do I set up a new project for this course from scratch using uv?', 'Which packages does the lesson tell me to add, and what is each one for?', "What's the safest way to keep my API key out of git, and what should go in the .gitignore file?", 'How do I launch Jupyter and verify that the OpenAI client is working in a notebook?']
1286
['Why does asking the LLM a course-specific question directly give a

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
from minsearch import Index

def text_search(query, num_results=5):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )

    index.fit(chunks)
    search_results = index.search(query, num_results=num_results)
    return search_results


In [14]:
import pandas as pd

ground_truth = pd.read_csv("../data/ground-truth.csv")
ground_truth = ground_truth.to_dict(orient="records")
q = ground_truth[0]["question"]

In [16]:
text_search(q)

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [26]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v1 = embed.encode(q1)

In [27]:
v1

array([-2.05820344e-02, -1.40458849e-02,  3.02994061e-02, -5.40378445e-02,
        7.18781100e-02, -2.79537512e-02, -5.03093823e-02, -1.27217287e-02,
        4.08207902e-02, -2.60037446e-02,  3.05458646e-02,  4.21485309e-02,
        8.09861910e-02, -6.93957355e-02, -1.30190518e-01, -6.39247860e-02,
        4.81059741e-02,  1.60095554e-02, -5.22432468e-02, -7.13635281e-02,
       -3.83859209e-03,  2.48125508e-02,  4.40211692e-02, -3.45579077e-02,
        1.52686257e-02,  7.61533350e-03,  5.38177679e-02,  1.18252557e-02,
        1.32434005e-02,  2.89461963e-02,  6.64912054e-03,  7.04788357e-02,
        6.19290508e-02,  2.11051780e-02, -7.33482889e-02,  2.84129213e-02,
       -3.72108635e-02,  6.22799314e-02, -4.86973136e-02,  4.49663910e-02,
       -2.59481060e-02,  2.04324269e-02,  1.79650458e-02,  1.02705602e-02,
        2.74898095e-03,  2.84324992e-02, -3.30318167e-02,  6.70969402e-02,
       -1.56520555e-02, -8.51907638e-02, -1.24307135e-01,  4.32504103e-02,
       -5.64433014e-02,  

In [39]:
content = [doc['content'] for doc in documents]
content[:3]



['# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language

In [40]:
chunks[:3]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [51]:
len(chunks)

295

In [42]:
content = [chunk['content'] for chunk in chunks]

In [43]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = content[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [48]:
scores = X.dot(v1)

In [50]:
len(scores)

295

In [55]:
from minsearch import VectorSearch

def vector_search(query, num_results=5):
    v = embed.encode(query)
    index = VectorSearch(
        keyword_fields=["filename"]
    )

    index.fit(X, chunks)
    search_results = index.search(v, num_results=num_results)
    return search_results


In [57]:
q = ground_truth[0]["question"]
vector_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [59]:
def compute_relevance(q, search_function):
    filename = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance


def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [60]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [61]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [62]:
hit_rate(relevance_total)

0.7583333333333333

In [63]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)
    

In [64]:
relevance_vector_search = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [65]:
mrr(relevance_vector_search)

0.5486111111111112

In [66]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [73]:
def compute_relevance(q, search_function, k):
    filename = q["filename"]
    results = search_function(query=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance


def compute_relevance_total(ground_truth, search_function, k):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, k)
        relevance_total.append(relevance)

    return relevance_total

In [74]:
for k in [1, 50, 100, 200]:
    relevance_hybrid = compute_relevance_total(ground_truth, hybrid_search, k)
    print(f"Hybrid search with k={k}:")
    print(f"MRR: {mrr(relevance_hybrid)}")

  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid search with k=1:
MRR: 0.6481944444444449


  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid search with k=50:
MRR: 0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid search with k=100:
MRR: 0.637916666666667


  0%|          | 0/360 [00:00<?, ?it/s]

Hybrid search with k=200:
MRR: 0.637916666666667
